In [ ]:
from pynq import Overlay, MMIO
from time import sleep
from datetime import datetime

class NEC_IR_Controller_24ch:
    def __init__(self, bitfile_path, tx_bases, btn_base,
                 command_dict=None):
        print(f"[DEBUG] Loading overlay from: {bitfile_path}")
        self.ol = Overlay(bitfile_path)
        self.ol.download()
        print("[DEBUG] Overlay loaded and downloaded.")

        if command_dict is None:
            command_dict = {
                '-': 0x07,
                '+': 0x15,
                '0': 0x16,
                '1': 0x0C,
                '2': 0x18,
                '3': 0x5E,
                '4': 0x08,
                '5': 0x1C,
                '6': 0x5A,
                '7': 0x42,
                '8': 0x52,
                '9': 0x4A,
                'A': 0xAA,
                'B': 0xBB,
                'C': 0xCC,
                'D': 0xDD,
                'E': 0xEE,
                'F': 0xF1
            }
        self.command_dict = command_dict
        
        print("[DEBUG] Initializing MMIO handles for TX channels...")
        self.txs = [MMIO(base, 0x10000) for base in tx_bases]
        print(f"[DEBUG] Initialized {len(self.txs)} TX MMIO handles.")
        self.btn = MMIO(btn_base, 0x10000)
        print("[DEBUG] Initialized BTN MMIO handle.")

        print("[DEBUG] Configuring GPIO directions for transmitters and button...")
        for tx in self.txs:
            tx.write(0x04, 0x00000000)  # ch1 as output
            tx.write(0x0C, 0x00000000)  # ch2 as output
        self.btn.write(0x04, 0x00000000)  # output
        print("[DEBUG] GPIO directions configured.")

        self.NEC_HOLD_TIME = 0.3      # 300 ms
        self.BTN_PULSE_WIDTH = 0.001  # 1 ms

        print("[DEBUG] Clearing transmitter and button states at init...")
        self.clear_all_tx()
        self._btn_low()
        print("[DEBUG] Initialization complete.")

    def clear_all_tx(self):
        print("[DEBUG] Clearing all transmitter lines")
        for tx in self.txs:
            tx.write(0x00, 0x00000000)  # CH1 low
            tx.write(0x08, 0x00000000)  # CH2 low

    def _btn_high(self):
        print("[DEBUG] Button HIGH pulse.")
        self.btn.write(0x00, 0x1)

    def _btn_low(self):
        print("[DEBUG] Button LOW.")
        self.btn.write(0x00, 0x0)

    def reverse_bits_8bit(self, value):
        value &= 0xFF
        reversed_val = 0
        for i in range(8):
            if (value >> i) & 1:
                reversed_val |= 1 << (7 - i)
        return reversed_val

    def assign_value(self, ch_idx, addr, cmd_char, reverse=False):
        if reverse:
            addr = self.reverse_bits_8bit(addr)
            hex_cmd = self.reverse_bits_8bit(self.command_dict[cmd_char])
        else:
            hex_cmd = self.command_dict[cmd_char]
        tx = self.txs[ch_idx]
        tx.write(0x00, addr & 0xFF)
        tx.write(0x08, hex_cmd & 0xFF)
        print(f"[DEBUG] TX{ch_idx} assigned: addr={hex(addr & 0xFF)}, cmd={hex(hex_cmd & 0xFF)}")

    def send_command_full(self, addresses, commands_str, reverse=False):
        ch_count = len(self.txs)
        assert len(addresses) == ch_count
        assert all(len(commands_str[0]) == len(cmd) for cmd in commands_str), "All command strings must have equal length"

        for idx in range(len(commands_str[0])):
            self.clear_all_tx()
            self._btn_low()
            sleep(self.BTN_PULSE_WIDTH)

            for ch in range(ch_count):
                cmd_char = commands_str[ch][idx]
                self.assign_value(ch, addresses[ch], cmd_char, reverse)

            print(f"[DEBUG] Pulsing button for commands {', '.join(cmd[idx] for cmd in commands_str)}...")
            self._btn_high()
            sleep(self.BTN_PULSE_WIDTH)
            self._btn_low()
            sleep(self.NEC_HOLD_TIME)
        self.clear_all_tx()
        print("[DEBUG] send_command_full finished.")

    def cleanup(self):
        print("[DEBUG] Cleaning up: clearing transmitters and button.")
        self.clear_all_tx()
        self._btn_low()
        print("[DEBUG] Cleanup complete.")

# Usage example for 24 channels:
if __name__ == "__main__":
    # Example consecutive base addresses (replace with your actual ones if not consecutive)
    base_start = 0x41210000
    tx_bases = [base_start + 0x10000 * i for i in range(24)]
    btn_base = 0x41200000

    ir = NEC_IR_Controller_24ch(
        "/home/xilinx/jupyter_notebooks/xilinx/overlays/own/design_1_wrapper.bit",
        tx_bases=tx_bases,
        btn_base=btn_base
    )

    try:
        start = datetime.now()
        # Each tx needs address and a command string of equal length
        tx_addresses =  [
            0x06,  # Channel 0: address 0x06
            0x07,  # Channel 1: address 0x07
            0x08,  # Channel 2: address 0x08
            0x09,  # Channel 3: address 0x09
            0x0A,  # Channel 4: address 0x0A
            0x0B,  # Channel 5: address 0x0B
            0x0C,  # Channel 6: address 0x0C
            0x0D,  # Channel 7: address 0x0D
            0x0E,  # Channel 8: address 0x0E
            0x0F,  # Channel 9: address 0x0F
            0x10,  # Channel 10: address 0x10
            0x11,  # Channel 11: address 0x11
            0x12,  # Channel 12: address 0x12
            0x13,  # Channel 13: address 0x13
            0x14,  # Channel 14: address 0x14
            0x15,  # Channel 15: address 0x15
            0x16,  # Channel 16: address 0x16
            0x17,  # Channel 17: address 0x17
            0x18,  # Channel 18: address 0x18
            0x19,  # Channel 19: address 0x19
            0x1A,  # Channel 20: address 0x1A
            0x1B,  # Channel 21: address 0x1B
            0x1C,  # Channel 22: address 0x1C
            0x1D   # Channel 23: address 0x1D
        ]
             # Example addresses
        cmd_strs = [
            "-0005+",  # Channel 0: 5
            "-0006+",  # Channel 1: 6
            "-0007+",  # Channel 2: 7
            "-0008+",  # Channel 3: 8
            "-0009+",  # Channel 4: 9
            "-0010+",  # Channel 5: 10
            "-0011+",  # Channel 6: 11
            "-0012+",  # Channel 7: 12
            "-0013+",  # Channel 8: 13
            "-0014+",  # Channel 9: 14
            "-0015+",  # Channel 10: 15
            "-0016+",  # Channel 11: 16
            "-0017+",  # Channel 12: 17
            "-0018+",  # Channel 13: 18
            "-0019+",  # Channel 14: 19
            "-0020+",  # Channel 15: 20
            "-0021+",  # Channel 16: 21
            "-0022+",  # Channel 17: 22
            "-0023+",  # Channel 18: 23
            "-0024+",  # Channel 19: 24
            "-0025+",  # Channel 20: 25
            "-0026+",  # Channel 21: 26
            "-0027+",  # Channel 22: 27
            "-0028+",  # Channel 23: 28
        ]
        ir.send_command_full(tx_addresses, cmd_strs, reverse=True)
        print(f"Elapsed: {datetime.now() - start}")

    except Exception as e:
        print(f"Error: {e}")
    finally:
        ir.cleanup()
        print("Cleanup complete")

[DEBUG] Loading overlay from: /home/xilinx/jupyter_notebooks/xilinx/overlays/own/design_1_wrapper.bit
[DEBUG] Overlay loaded and downloaded.
[DEBUG] Initializing MMIO handles for TX channels...
[DEBUG] Initialized 24 TX MMIO handles.
[DEBUG] Initialized BTN MMIO handle.
[DEBUG] Configuring GPIO directions for transmitters and button...
[DEBUG] GPIO directions configured.
[DEBUG] Clearing transmitter and button states at init...
[DEBUG] Clearing all transmitter lines
[DEBUG] Button LOW.
[DEBUG] Initialization complete.
[DEBUG] Clearing all transmitter lines
[DEBUG] Button LOW.
[DEBUG] TX0 assigned: addr=0x60, cmd=0xe0
[DEBUG] TX1 assigned: addr=0xe0, cmd=0xe0
[DEBUG] TX2 assigned: addr=0x10, cmd=0xe0
[DEBUG] TX3 assigned: addr=0x90, cmd=0xe0
[DEBUG] TX4 assigned: addr=0x50, cmd=0xe0
[DEBUG] TX5 assigned: addr=0xd0, cmd=0xe0
[DEBUG] TX6 assigned: addr=0x30, cmd=0xe0
[DEBUG] TX7 assigned: addr=0xb0, cmd=0xe0
[DEBUG] TX8 assigned: addr=0x70, cmd=0xe0
[DEBUG] TX9 assigned: addr=0xf0, cmd=0x